In [ ]:
# Colab setup
import os
import sys
from pathlib import Path

DRIVE_PROJECT = "Colab Notebooks/DDL-Diffusion/code"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q diffusers transformers datasets
    from google.colab import drive

    drive.mount("/content/drive")

    drive_root = Path("/content/drive/MyDrive")
    project_dir = drive_root / DRIVE_PROJECT

    if not (project_dir / "scheduler.py").is_file():
        raise RuntimeError(
            f"no scheduler.py under {project_dir} — upload the .py files to that "
            f"folder, or fix DRIVE_PROJECT above"
        )
    os.chdir(project_dir)
# else:
    # Local kernels don't agree on the starting cwd: Jupyter uses the notebook's
    # own directory, PyCharm/VS Code often use the project root. Walk up until we
    # find code/ so the bare `from scheduler import ...` imports and the relative
    # .pt paths resolve either way.
    # start = Path.cwd().resolve()
    # for base in (start, *start.parents):
    #     target = base if base.name == "code" else base / "code"
    #     if (target / "scheduler.py").is_file():
    #         os.chdir(target)
    #         break
    # else:
    #     raise RuntimeError(f"couldn't locate the repo's code/ directory from {start}")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("colab:", IN_COLAB)
print("cwd  :", os.getcwd())

In [ ]:
import torch
import datasets
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms as T
from diffusers import AutoencoderKL
from tqdm import tqdm
from transformers import CLIPTokenizer, CLIPTextModel

In [ ]:
device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
device

In [ ]:
SEED = 42   # single source of truth: seeds training and the preview sampler

torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True

In [ ]:
poke_data = datasets.load_dataset("diffusers/pokemon-gpt4-captions")

In [ ]:
print(poke_data)
print(poke_data["train"][0])

In [ ]:
sample = poke_data["train"][0]
plt.imshow(sample["image"])
plt.show()
print(sample["text"])

In [ ]:
# Resolution and augmentation config.
#
# Run 3 fixed run 2's memorization (held-out loss 1.5157 -> 0.6193, conditioning went from
# a lookup table to genuine) but the samples came out blurry. Measured cause: augmenting
# each image into 8 views made 59.9% of the target variance UNPREDICTABLE from the caption
# — the model cannot know which framing it is being asked for, so an L2 objective is
# minimized by predicting the AVERAGE of the 8 views, and decoding an average is blur.
#
#   config                  views/img   view noise
#   1 view (run 1)                  1        0.0%
#   +flip  (run 2)                  2       38.6%
#   4 crops, no flip                4       38.8%
#   4 crops + flip (run 3)          8       59.9%
#
# Flip alone accounts for 38.6%, so trimming crops does not fix it. Run 4 keeps all 8
# views and instead CONDITIONS on the view (crop box + flip) the way SDXL does. The
# ambiguity becomes signal: ask for the canonical view at inference and the
# caption->image mapping is single-valued again.
RESOLUTION = 256
LATENT_SIZE = RESOLUTION // 8       # the SD VAE downsamples by 8
VAE_SCALE = 0.18215

CROPS = 4                           # crop 0 is the deterministic centre crop; 1..N-1 random
CROP_SCALE = (0.8, 1.0)
HFLIP = True
CAPTION_VARIANTS = 2                # 1 = original only, 2 = + name-stripped
VIEW_COND = True                    # micro-condition on the crop box + flip flag

from precompute import build_preprocess, crop_with_params, VIEW_DIM, CANONICAL_VIEW

preprocess = build_preprocess(RESOLUTION, crop=False)          # canonical view, for display
test_image = preprocess(sample["image"])
print(test_image.shape, "-> latents will be", (4, LATENT_SIZE, LATENT_SIZE))
print(f"view conditioning: {VIEW_COND}   view_dim {VIEW_DIM}   canonical {CANONICAL_VIEW}")
print(f"planned latents: {len(poke_data['train'])} x {CROPS} crops x {2 if HFLIP else 1} flips "
      f"= {len(poke_data['train']) * CROPS * (2 if HFLIP else 1)}")

# What the crops look like, with the view vector that will be fed to the U-Net alongside.
fig, axes = plt.subplots(1, 5, figsize=(15, 3.6))
for k in range(5):
    img, view = crop_with_params(sample["image"], RESOLUTION, crop=k > 0, scale=CROP_SCALE)
    axes[k].imshow(((img + 1) / 2).permute(1, 2, 0).numpy())
    axes[k].set_title(("crop 0 (centre)" if k == 0 else f"random crop {k}") +
                      "\n" + str([round(v, 2) for v in view]), fontsize=8)
    axes[k].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse").to(device)
vae.eval()
vae.requires_grad_(False)

In [ ]:
x = preprocess(poke_data["train"][0]["image"]).unsqueeze(0).to(device)

with torch.no_grad():
    # .mode() not .sample(). We freeze these latents to disk, so a single stochastic
    # posterior draw would bake one fixed noise pattern into every training target
    # forever. The mode is deterministic.
    latent = vae.encode(x).latent_dist.mode() * VAE_SCALE

print(latent.shape)
print(latent.min(), latent.max(), latent.mean(), latent.std())
print("per-channel mean:", [round(v, 3) for v in latent.mean(dim=(0, 2, 3)).tolist()])
print("\nThose per-channel means are far from 0 — the next cell normalizes them away.")
print("Run 1 skipped that and the model never learned to generate the offset.")

In [ ]:
# Encode the WHOLE dataset -> latents.pt + latent_stats.pt
#
# One VAE pass per (crop, flip) combination. Four index arrays come out alongside:
#
#   group_ids   : source image. train.make_split splits on this, so every crop and flip of
#                 one Pokemon stays on the same side of the train/val boundary.
#   crop_ids    : which crop. Validation uses crop 0 only, so the number is comparable
#                 across runs and not diluted by augmentation.
#   flip_ids    : mirrored or not.
#   view_params : (N, 5) = [top, left, height, width, flip], the box normalized against the
#                 source image. NEW in run 4 — this is the micro-conditioning that stops
#                 the augmentation from being unpredictable noise.
from precompute import normalize_per_channel

def encode_pass(crop, flip, desc):
    lats, views = [], []
    for i in tqdm(range(0, len(poke_data["train"]), 16), desc=desc, leave=False):
        pairs = [crop_with_params(im, RESOLUTION, crop=crop, scale=CROP_SCALE, flip=flip)
                 for im in poke_data["train"][i : i + 16]["image"]]
        imgs = torch.stack([p[0] for p in pairs]).to(device)
        views.extend(p[1] for p in pairs)
        with torch.no_grad():
            lat = vae.encode(imgs).latent_dist.mode() * VAE_SCALE
        lats.append(lat.cpu())   # .cpu() is critical — don't hoard on device
    return torch.cat(lats, dim=0), torch.tensor(views, dtype=torch.float32)

N_IMAGES = len(poke_data["train"])
flips = (False, True) if HFLIP else (False,)

chunks, vchunks, group_ids, crop_ids, flip_ids = [], [], [], [], []
for crop_i in range(CROPS):
    torch.manual_seed(SEED + crop_i)          # reproducible crop geometry
    for flip in flips:
        tag = f"crop {crop_i}{' flip' if flip else ''}"
        lat, vw = encode_pass(crop_i > 0, flip, tag)
        chunks.append(lat); vchunks.append(vw)
        group_ids.append(torch.arange(N_IMAGES))
        crop_ids.append(torch.full((N_IMAGES,), crop_i))
        flip_ids.append(torch.full((N_IMAGES,), int(flip)))
        print(f"  {tag:14s} done   view e.g. {[round(v, 2) for v in vw[0].tolist()]}")

raw_latents = torch.cat(chunks, dim=0)
view_params = torch.cat(vchunks, dim=0)
group_ids = torch.cat(group_ids).long()
crop_ids = torch.cat(crop_ids).long()
flip_ids = torch.cat(flip_ids).long()

print(f"\nraw {tuple(raw_latents.shape)}   views {tuple(view_params.shape)}")
print("  per-chan mean:", [round(v, 3) for v in raw_latents.mean(dim=(0, 2, 3)).tolist()])
print("  per-chan std :", [round(v, 3) for v in raw_latents.std(dim=(0, 2, 3)).tolist()])

latents_tensor, LAT_MEAN, LAT_STD = normalize_per_channel(raw_latents)
print("\nnormalized")
print("  per-chan mean:", [round(v, 4) for v in latents_tensor.mean(dim=(0, 2, 3)).tolist()], "(want ~0)")
print("  per-chan std :", [round(v, 4) for v in latents_tensor.std(dim=(0, 2, 3)).tolist()], "(want ~1)")
print(f"  overall mean {latents_tensor.mean():+.5f}  std {latents_tensor.std():.5f}  "
      f"abs max {latents_tensor.abs().max():.2f}")

# How much of the target is unpredictable from the caption alone? This is the number
# view conditioning exists to neutralize — print it so run 4 can be compared to run 3's
# 59.9% directly.
_X = latents_tensor.flatten(1).double()
_s = torch.zeros(N_IMAGES, _X.shape[1], dtype=torch.float64)
_c = torch.zeros(N_IMAGES, dtype=torch.float64)
_s.index_add_(0, group_ids, _X); _c.index_add_(0, group_ids, torch.ones(len(_X), dtype=torch.float64))
_gm = _s / _c[:, None]
_w = ((_X - _gm[group_ids]) ** 2).mean().item()
_t = ((_X - _X.mean(0)) ** 2).mean().item()
print(f"\nview noise (variance not predictable from the caption): {_w/_t:.1%}   (run 3: 59.9%)")
print("  view conditioning is what turns this from noise into signal.")
del _X, _s, _c, _gm

torch.save(latents_tensor, "latents.pt")
del raw_latents, chunks
print("\nsaved latents.pt")

In [ ]:
# Decode saved latents back to images — a round-trip check on the normalization,
# and a look at what the crop augmentation actually feeds the model.
#
# LAT_MEAN / LAT_STD are still in memory from the cell above; latent_stats.pt is written
# after the captions, once the index arrays exist.
from sample import latents_to_images

latents = torch.load("latents.pt")

# Row layout is [crop0, crop0-flip, crop1, crop1-flip, ...], so image 0 under each
# (crop, flip) pass sits at pass_index * N_IMAGES.
n_passes = CROPS * (2 if HFLIP else 1)
picks = [p * N_IMAGES for p in range(min(n_passes, 4))]

fig, axes = plt.subplots(1, len(picks) + 1, figsize=(3.2 * (len(picks) + 1), 3.8))
for ax, row in zip(axes, picks):
    img = latents_to_images(latents[row : row + 1].to(device),
                            vae, lat_mean=LAT_MEAN, lat_std=LAT_STD)[0].cpu()
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(f"row {row}  crop {int(crop_ids[row])}"
                 f"{' flip' if int(flip_ids[row]) else ''}", fontsize=9)

# Forgetting lat_mean / lat_std is THE mistake to make from here on: the model's latent
# space is no longer the VAE's, so decoding without undoing the shift gives off-colour
# mush. Worth seeing once so you recognise it in a sample later.
bad = latents_to_images(latents[0:1].to(device), vae)[0].cpu()
axes[-1].imshow(bad.permute(1, 2, 0).numpy())
axes[-1].set_title("lat_mean / lat_std NOT undone", fontsize=9)

for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
CLIP_NAME = "openai/clip-vit-large-patch14"
tokenizer = CLIPTokenizer.from_pretrained(CLIP_NAME)
text_encoder = CLIPTextModel.from_pretrained(CLIP_NAME).to(device)
text_encoder.eval()
text_encoder.requires_grad_(False)




In [ ]:
# Verify shape (1, 77, 768).
test_caption = poke_data["train"][0]["text"]

tokens = tokenizer(
    test_caption,
    padding="max_length",
    max_length=77,
    truncation=True,
    return_tensors="pt",
)
input_ids = tokens.input_ids.to(device)

with torch.no_grad():
    test_emb = text_encoder(input_ids).last_hidden_state

print(test_emb.shape)
print(test_emb.min(), test_emb.max(), test_emb.mean())

In [ ]:
# Encode caption VARIANTS -> embeddings.pt, and write latent_stats.pt
#
# Run 3 confirmed this worked: `stripped` came within 0.1% of `matched` at every
# timestep, meaning removing the Pokemon name costs the model nothing — the name-as-
# lookup-key habit from run 2 is gone. Keep it.
#
# Captions are stored ONCE PER (image, variant) and joined to latents by group id, not
# duplicated per crop — a 77x768 embedding is 237 KB, so duplicating across 8 views would
# push embeddings.pt past 1.5 GB.
from precompute import build_caption_variants, strip_pokemon_name

all_captions = [row["text"] for row in poke_data["train"]]
flat_captions, caption_groups = build_caption_variants(all_captions, CAPTION_VARIANTS)

n_changed = sum(1 for c in all_captions if strip_pokemon_name(c)[1])
print(f"{len(all_captions)} images x {CAPTION_VARIANTS} variants = {len(flat_captions)} captions")
print(f"name found and stripped in {n_changed}/{len(all_captions)} ({n_changed/len(all_captions):.0%})\n")
for c in all_captions[:2]:
    print(f"  orig: {c}")
    print(f"  var : {strip_pokemon_name(c)[0]}\n")

all_input_ids = tokenizer(
    flat_captions,
    padding="max_length",
    max_length=77,
    truncation=True,
    return_tensors="pt",
).input_ids

all_embeddings = []
for i in tqdm(range(0, len(flat_captions), 32), desc="CLIP encode"):
    with torch.no_grad():
        emb = text_encoder(all_input_ids[i : i + 32].to(device)).last_hidden_state
    all_embeddings.append(emb.cpu())

embeddings_tensor = torch.cat(all_embeddings, dim=0)
print(f"\nembeddings {tuple(embeddings_tensor.shape)}")
torch.save(embeddings_tensor, "embeddings.pt")

torch.save({
    "mean": LAT_MEAN, "std": LAT_STD, "scale": VAE_SCALE,
    "resolution": RESOLUTION, "latent_size": latents_tensor.shape[-1],
    "latent_channels": latents_tensor.shape[1], "n_images": N_IMAGES,
    "crops": CROPS, "crop_scale": CROP_SCALE, "hflip": HFLIP,
    "caption_variants": CAPTION_VARIANTS, "normalized": True,
    "group_ids": group_ids, "crop_ids": crop_ids, "flip_ids": flip_ids,
    "caption_groups": caption_groups, "seed": SEED,
    "view_params": view_params, "view_dim": VIEW_DIM,
    "canonical_view": torch.tensor(CANONICAL_VIEW), "view_cond": VIEW_COND,
}, "latent_stats.pt")
print("saved latent_stats.pt (index arrays + view params + normalization stats)")

In [ ]:
# Encode the "" -> uncond_embedding.pt
# Used for Classifier-Free Guidance (CFG) during training and inference.
uncond_ids = tokenizer(
    "",
    padding="max_length",
    max_length=77,
    truncation=True,
    return_tensors="pt",
).input_ids.to(device)

with torch.no_grad():
    uncond_embedding = text_encoder(uncond_ids).last_hidden_state

print(uncond_embedding.shape)   # (1, 77, 768)
torch.save(uncond_embedding.cpu(), "uncond_embedding.pt")

In [ ]:
# autoreload makes Jupyter pick up edits to .py files without restarting the kernel.
# %load_ext autoreload
# %autoreload 2
from scheduler import NoiseScheduler

In [ ]:
# The noise schedule. Two changes from run 1, and this cell is where you verify them.
#
# ZERO TERMINAL SNR.  Run 1 had alphas_cumprod[-1] = 0.00158, so
# sqrt(alphas_cumprod[-1]) = 0.0397 and training at t=999 still leaked 0.0397 * x_0
# into x_t. Because the raw latents had per-channel means up to +1.50, that leak was
# a per-channel DC offset — detectable at roughly 4 sigma once averaged over a
# channel's 1024 spatial positions. So the model learned to READ the output's DC
# level off its input instead of GENERATING it. At inference x_T = torch.randn has
# per-channel mean exactly 0, the cue is gone, and the channels collapse together.
# That is the flaw in "Common Diffusion Noise Schedules and Sample Steps are Flawed"
# (Lin et al.); Algorithm 1 there rescales the schedule so alphas_cumprod[-1] = 0.
#
# V-PREDICTION.  Required once alphas_cumprod[-1] = 0: recovering x_0 from an eps
# prediction needs (x_t - sqrt(1-ab) eps) / sqrt(ab), which divides by zero. The v
# route, x_0 = sqrt(ab) x_t - sqrt(1-ab) v, is an exact rotation and finite
# everywhere. It also flattens the loss across t — see the next few cells.
scheduler = NoiseScheduler(zero_terminal_snr=True, prediction_type="v").to(device)
run1 = NoiseScheduler(zero_terminal_snr=False, prediction_type="eps")

rows = [
    ("alphas_cumprod[0]",        run1.alphas_cumprod[0].item(),        scheduler.alphas_cumprod[0].item(),        "want ~1.0"),
    ("alphas_cumprod[-1]",       run1.alphas_cumprod[-1].item(),       scheduler.alphas_cumprod[-1].item(),       "want exactly 0"),
    ("sqrt(alphas_cumprod[-1])", run1.sqrt_alphas_cumprod[-1].item(),  scheduler.sqrt_alphas_cumprod[-1].item(),  "<- the leak"),
    ("betas[0]",                 run1.betas[0].item(),                 scheduler.betas[0].item(),                 ""),
    ("betas[-1]",                run1.betas[-1].item(),                scheduler.betas[-1].item(),                "1.0 is expected here"),
]
print(f"{'':26s} {'run 1':>12} {'run 2':>12}")
for name, a, b, note in rows:
    print(f"{name:26s} {a:12.7f} {b:12.7f}   {note}")

assert scheduler.alphas_cumprod[-1].item() == 0.0, "terminal SNR is not zero"
print("\nx_T is now pure noise by construction — exactly what inference samples from.")

In [ ]:
# Noise one real Pokemon latent across the schedule and decode each step.
#
# The last panel is the one that matters: with zero terminal SNR, t=999 is EXACTLY
# pure noise, with no trace of the Pokemon left. Run 1's schedule kept 3.97% of the
# signal there, and the model quietly came to depend on it.
latents = torch.load("latents.pt")
stats = torch.load("latent_stats.pt")
x_0 = latents[0:1].to(device)

noise = torch.randn_like(x_0)
timesteps_to_show = [0, 250, 500, 750, 999]

fig, axes = plt.subplots(1, len(timesteps_to_show), figsize=(3.2 * len(timesteps_to_show), 3.8))
for ax, t_val in zip(axes, timesteps_to_show):
    t = torch.full((1,), t_val, device=device, dtype=torch.long)
    x_t = scheduler.q_sample(x_0, t, noise)
    img = latents_to_images(x_t, vae, lat_mean=stats["mean"], lat_std=stats["std"])[0].cpu()
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(f"t = {t_val}\nsqrt(ab) = {scheduler.sqrt_alphas_cumprod[t_val].item():.4f}", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Test the U-Net.
#
# Run 3's architecture was right: base_channels=64 / 2 blocks / no top-res self-attention,
# 17.4M params against 24.6M training scalars (0.71x). It fixed memorization. Keep it.
#
# The one addition is view_dim=5 micro-conditioning (crop box + flip), which costs 266k
# params and is zero-initialised so a fresh model behaves exactly as if it were absent.
from unet import UNet

BASE_CHANNELS = 64
NUM_RES_BLOCKS = 2
TOP_SELF_ATTN = False       # 50% of the forward pass at 64x64; SD has none there either
DROPOUT = 0.1
VIEW_DIM_USED = VIEW_DIM if VIEW_COND else 0

model = UNet(base_channels=BASE_CHANNELS, num_res_blocks=NUM_RES_BLOCKS,
             top_self_attn=TOP_SELF_ATTN, dropout=DROPOUT, view_dim=VIEW_DIM_USED).to(device)

B = 2
x = torch.randn(B, 4, LATENT_SIZE, LATENT_SIZE, device=device)
t = torch.randint(0, 1000, (B,), device=device, dtype=torch.long)
c = torch.randn(B, 77, 768, device=device)
v = torch.tensor([CANONICAL_VIEW] * B, device=device) if VIEW_COND else None

model.eval()
with torch.no_grad():
    out = model(x, t, c, v)
model.train()

print("Output shape :", tuple(out.shape))
print("Output std   :", out.std().item(), "(0.0 by design — zero-init out_conv)")

n_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {n_params:,}   (run 1: 12,658,628   run 2: 35,540,836   run 3: 17,368,644)")
print(f"Full checkpoint on disk: ~{n_params * 16 / 1e6:.0f} MB   slim: ~{n_params * 8 / 1e6:.0f} MB")

In [ ]:
from torch.utils.data import DataLoader, Subset
from train import make_split, LatentCaptionDataset

latents_tensor = torch.load("latents.pt")             # per-channel normalized
embeddings_tensor = torch.load("embeddings.pt")       # (n_images * variants, 77, 768)
uncond_embedding = torch.load("uncond_embedding.pt")  # (1, 77, 768) CLIP-encoded ""
stats = torch.load("latent_stats.pt")

LAT_MEAN, LAT_STD = stats["mean"], stats["std"]
LATENT_SIZE = stats["latent_size"]
LATENT_SHAPE = (stats["latent_channels"], LATENT_SIZE, LATENT_SIZE)
N_IMAGES = stats["n_images"]
group_ids, crop_ids = stats["group_ids"], stats["crop_ids"]
caption_groups = stats["caption_groups"]
VIEW_COND = bool(stats.get("view_cond", False))
view_params = stats["view_params"] if VIEW_COND else None
CANON = stats["canonical_view"]

print("latents   :", tuple(latents_tensor.shape), latents_tensor.dtype)
print("embeddings:", tuple(embeddings_tensor.shape), f"({stats['caption_variants']} variants/image)")
print("view cond :", VIEW_COND, "" if not VIEW_COND else f"dim {stats['view_dim']}  canonical {CANON.tolist()}")
print("per-chan mean:", [round(v, 4) for v in latents_tensor.mean(dim=(0, 2, 3)).tolist()], "(want ~0)")
print("per-chan std :", [round(v, 4) for v in latents_tensor.std(dim=(0, 2, 3)).tolist()], "(want ~1)")

# Held-out split, on GROUPS. Every crop and flip of one Pokemon stays together.
VAL_FRAC = 0.1
train_idx, val_idx = make_split(group_ids, val_frac=VAL_FRAC, seed=0)

full_dataset = LatentCaptionDataset(
    latents_tensor, group_ids, embeddings_tensor, caption_groups,
    view_params=view_params, sample_variant=True,
)
caption_table = full_dataset.caption_table      # (n_images, n_variants) -> embedding row


def caption_emb(image_ids, variant=0):
    """(B, 77, 768) for a list of SOURCE IMAGE ids. Variant 0 is the original caption."""
    idx = caption_table[torch.as_tensor(image_ids, dtype=torch.long), variant]
    return embeddings_tensor[idx]


def caption_emb_for_rows(rows, variant=0):
    """(B, 77, 768) for LATENT ROW indices."""
    return caption_emb(group_ids[torch.as_tensor(rows, dtype=torch.long)], variant)


def views_for_rows(rows):
    """(B, view_dim) actual view params for LATENT ROW indices, or None if unused."""
    return None if view_params is None else view_params[torch.as_tensor(rows, dtype=torch.long)]


def canonical_views(n):
    """(n, view_dim) the full-frame unflipped view — what you ask for at inference."""
    return None if not VIEW_COND else CANON.unsqueeze(0).repeat(n, 1)


BATCH_SIZE = 32
train_loader = DataLoader(
    Subset(full_dataset, train_idx.tolist()),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    drop_last=True,
)

# Validation uses crop 0 only (both flips), so the number is comparable to run 3's and not
# diluted by augmentation. Views are the ACTUAL ones for those rows — we are measuring how
# well the model predicts that specific latent.
val_eval_rows = val_idx[crop_ids[val_idx] == 0]
val_x0, val_ctx = latents_tensor[val_eval_rows], caption_emb_for_rows(val_eval_rows)
val_view = views_for_rows(val_eval_rows)

tr_eval_pool = train_idx[crop_ids[train_idx] == 0]
_g = torch.Generator().manual_seed(0)
tr_eval_rows = tr_eval_pool[torch.randperm(len(tr_eval_pool), generator=_g)[: len(val_eval_rows)]]
tr_x0, tr_ctx = latents_tensor[tr_eval_rows], caption_emb_for_rows(tr_eval_rows)
tr_view = views_for_rows(tr_eval_rows)

uncond_embedding = uncond_embedding.to(device)
print(f"\nimages     : {len(torch.unique(group_ids[train_idx]))} train / {len(torch.unique(group_ids[val_idx]))} val")
print(f"latent rows: {len(train_idx)} train / {len(val_idx)} val")
print(f"eval slices: {len(tr_eval_rows)} train / {len(val_eval_rows)} val  (crop 0 only)")
print(f"batches/epoch: {len(train_loader)}")
print(f"\nparams-per-training-scalar: "
      f"{sum(p.numel() for p in model.parameters()) / (len(train_idx) * LATENT_SHAPE[0] * LATENT_SIZE ** 2):.2f}x"
      f"   (run 2: 5.78x memorized, run 3: 0.71x did not)")

In [ ]:
from train import build_ema, make_optimizer, make_lr_schedule, pick_amp_dtype

NUM_EPOCHS = 150
WARMUP_STEPS = 500
LR = 2e-4
WEIGHT_DECAY = 0.05
EMA_DECAY = 0.999    # was 0.9999. Run 3's best checkpoint landed at epoch 34, where the
                     # 0.9999 ramp still lagged the live weights by 8% — extra blur on the
                     # exact checkpoint we ended up sampling from.
EMA_WARMUP = True
CFG_DROPOUT = 0.15
GRAD_CLIP = 1.0
MIN_SNR_GAMMA = None # correct for v-prediction; see RUNS.md run-2 corrections

amp_dtype, AMP = pick_amp_dtype("auto", device=device)

TOTAL_STEPS = NUM_EPOCHS * len(train_loader)

model = UNet(base_channels=BASE_CHANNELS, num_res_blocks=NUM_RES_BLOCKS,
             top_self_attn=TOP_SELF_ATTN, dropout=DROPOUT, view_dim=VIEW_DIM_USED).to(device)
noise_scheduler = NoiseScheduler(zero_terminal_snr=True, prediction_type="v").to(device)
ema_model = build_ema(model)
optimizer = make_optimizer(model, lr=LR, weight_decay=WEIGHT_DECAY)
lr_scheduler = make_lr_schedule(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=TOTAL_STEPS,
)

scaler = torch.amp.GradScaler("cuda") if (amp_dtype is torch.float16 and device == "cuda") else None

MODEL_CONFIG = {
    "base_channels": BASE_CHANNELS, "num_res_blocks": NUM_RES_BLOCKS,
    "top_self_attn": TOP_SELF_ATTN, "dropout": DROPOUT, "view_dim": VIEW_DIM_USED,
    "prediction_type": "v", "zero_terminal_snr": True,
    "latent_shape": LATENT_SHAPE, "resolution": stats["resolution"],
    "min_snr_gamma": MIN_SNR_GAMMA, "batch_size": BATCH_SIZE, "lr": LR,
    "weight_decay": WEIGHT_DECAY, "cfg_dropout": CFG_DROPOUT, "ema_decay": EMA_DECAY,
    "crops": stats["crops"], "hflip": stats["hflip"],
    "caption_variants": stats["caption_variants"],
}

print(f"Trainable params     : {sum(p.numel() for p in model.parameters()):,}")
print(f"Total steps          : {TOTAL_STEPS:,}  ({NUM_EPOCHS} epochs x {len(train_loader)} batches)")
print(f"Sample presentations : {TOTAL_STEPS * BATCH_SIZE:,}")
print(f"Peak LR              : {LR}   weight decay {WEIGHT_DECAY}   dropout {DROPOUT}")
print(f"EMA decay            : {EMA_DECAY}")
print(f"View conditioning    : {VIEW_COND} (dim {VIEW_DIM_USED})")
print(f"AMP                  : {AMP}   (GradScaler: {scaler is not None})")

In [ ]:
from train import train_step

smoke_losses = []
data_iter = iter(train_loader)

for i in range(5):
    x_0, context, view = next(data_iter)
    loss = train_step(
        model=model,
        ema_model=ema_model,
        noise_scheduler=noise_scheduler,
        optimizer=optimizer,
        lr_scheduler=lr_scheduler,
        x_0=x_0.to(device),
        context=context.to(device),
        uncond_embedding=uncond_embedding,
        cfg_dropout_prob=CFG_DROPOUT,
        grad_clip=GRAD_CLIP,
        ema_decay=EMA_DECAY,
        min_snr_gamma=MIN_SNR_GAMMA,
        view=view.to(device) if VIEW_COND else None,
        amp_dtype=amp_dtype,
        scaler=scaler,
        step=i if EMA_WARMUP else None,
    )
    smoke_losses.append(loss)
    print(f"step {i}: loss = {loss:.4f}")

# Test passes if the losses are finite and the first is ~1.0. E[v^2] = 1 for unit-variance
# latents at every t, and out_conv is zero-init, so a healthy run starts exactly at the
# predict-zero baseline. Far from 1.0 means the latents are not normalized or the
# scheduler is misconfigured.
print(f"\nfirst loss {smoke_losses[0]:.4f}  (want ~1.0 — the predict-zero baseline)")
print(f"view batch shape {tuple(view.shape)}  e.g. {[round(v, 2) for v in view[0].tolist()]}")

# Re-run the cell ABOVE before real training so the optimizer and LR schedule start clean.

In [ ]:
from pathlib import Path

from tqdm.auto import tqdm
from train import train_step, validation_loss, save_checkpoint, load_checkpoint
from sample import sample_to_image

CHECKPOINT_DIR = "checkpoints"
LOG_EVERY = 50
VAL_EVERY = 2
RESUME_PATH = None               # path to last.pt, or None to start fresh

# Run 3 saved only best.pt + last.pt, and best.pt (epoch 34, chosen by lowest val MSE) was
# the blurriest checkpoint of the run — MSE is minimized by predicting the conditional
# MEAN, so early stopping on it actively selects for blur. Keep best.pt as the divergence
# detector, but also snapshot a few milestones so the winner can be picked by LOOKING.
KEEP_BEST = True
MILESTONES = [30, 60, 100, 150]  # slim snapshots, ~139 MB each
FULL_EVERY = 50                  # last.pt, full state, for resuming

PREVIEW = True
PREVIEW_EVERY = 10
PREVIEW_STEPS = 50
PREVIEW_CFG = 2.0
PREVIEW_CFG_RESCALE = 0.7
PREVIEW_IMAGES = [1, 2, 3]       # SOURCE IMAGE ids (0's caption is heavily templated)
PREVIEW_NOVEL = "a red fire dragon pokemon with large wings"   # generalization, live
PREVIEW_DIR = "previews"


@torch.no_grad()
def _clip_encode(prompts):
    """Same pipeline as the training captions. Defined here so previews can mix a training
    caption with an unseen prompt in one strip."""
    ids = tokenizer(prompts, padding="max_length", max_length=77,
                    truncation=True, return_tensors="pt").input_ids.to(device)
    return text_encoder(ids).last_hidden_state.cpu()


# 3 training captions + 1 unseen prompt, so every preview shows both memorization and
# generalization side by side. Run 3's previews only had training captions, which is why
# the blur was not obviously a generalization problem until the eval section ran.
_prev_cond = torch.cat([caption_emb(PREVIEW_IMAGES), _clip_encode([PREVIEW_NOVEL])], dim=0)
_prev_titles = [f"train #{i}" for i in PREVIEW_IMAGES] + ["UNSEEN"]


def make_preview(epoch: int) -> None:
    """DDIM samples from the EMA model: save a PNG strip and show it inline."""
    from torchvision.utils import make_grid, save_image

    g = torch.Generator(device=device).manual_seed(SEED)
    imgs = sample_to_image(
        model=ema_model,
        scheduler=noise_scheduler,
        vae=vae,
        cond_emb=_prev_cond.to(device),
        uncond_emb=uncond_embedding,
        method="ddim",
        guidance_scale=PREVIEW_CFG,
        guidance_rescale=PREVIEW_CFG_RESCALE,
        num_steps=PREVIEW_STEPS,
        latent_shape=LATENT_SHAPE,
        lat_mean=LAT_MEAN,
        lat_std=LAT_STD,
        view=canonical_views(_prev_cond.shape[0]).to(device) if VIEW_COND else None,
        generator=g,
    ).cpu()

    out_path = Path(PREVIEW_DIR) / f"epoch_{epoch}.png"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    grid = make_grid(imgs, nrow=imgs.shape[0])
    save_image(grid, str(out_path))

    plt.figure(figsize=(3.2 * imgs.shape[0], 3.6))
    plt.imshow(grid.permute(1, 2, 0).numpy())
    plt.title(f"epoch {epoch}   {' | '.join(_prev_titles)}   (CFG {PREVIEW_CFG})")
    plt.axis("off")
    plt.show()


start_step = 0
start_epoch = 0
loss_history: list[float] = []
val_history: list[tuple[int, float, float]] = []   # (step, train, val)
best_val = float("inf")
best_epoch = -1

if RESUME_PATH is not None:
    start_step, start_epoch, loss_history = load_checkpoint(
        RESUME_PATH, model, ema_model, optimizer, lr_scheduler,
        map_location=device, scaler=scaler,
    )
    model.to(device); ema_model.to(device)
    print(f"resumed from {RESUME_PATH}: step={start_step}, epoch={start_epoch}")

global_step = start_step
running_loss = 0.0
running_count = 0


def _ckpt(name, slim):
    save_checkpoint(
        f"{CHECKPOINT_DIR}/{name}", model, ema_model, optimizer, lr_scheduler,
        step=global_step, epoch=epoch + 1, loss_history=loss_history,
        scaler=scaler, config=MODEL_CONFIG, slim=slim,
    )


for epoch in range(start_epoch, NUM_EPOCHS):

    pbar = tqdm(train_loader, desc=f"epoch {epoch+1}/{NUM_EPOCHS}", leave=False)
    for x_0, context, view in pbar:
        loss = train_step(
            model=model,
            ema_model=ema_model,
            noise_scheduler=noise_scheduler,
            optimizer=optimizer,
            lr_scheduler=lr_scheduler,
            x_0=x_0.to(device),
            context=context.to(device),
            uncond_embedding=uncond_embedding,
            cfg_dropout_prob=CFG_DROPOUT,
            grad_clip=GRAD_CLIP,
            ema_decay=EMA_DECAY,
            min_snr_gamma=MIN_SNR_GAMMA,
            view=view.to(device) if VIEW_COND else None,
            amp_dtype=amp_dtype,
            scaler=scaler,
            step=global_step if EMA_WARMUP else None,
        )

        loss_history.append(loss)
        running_loss += loss
        running_count += 1
        global_step += 1

        if global_step % LOG_EVERY == 0:
            pbar.set_postfix(loss=f"{running_loss / running_count:.4f}",
                             lr=f"{optimizer.param_groups[0]['lr']:.2e}")
            running_loss = 0.0
            running_count = 0

    if (epoch + 1) % VAL_EVERY == 0:
        vl = validation_loss(ema_model, noise_scheduler, val_x0, val_ctx,
                             min_snr_gamma=MIN_SNR_GAMMA, view=val_view)["weighted"]
        tl = validation_loss(ema_model, noise_scheduler, tr_x0, tr_ctx,
                             min_snr_gamma=MIN_SNR_GAMMA, view=tr_view)["weighted"]
        val_history.append((global_step, tl, vl))
        flag = ""
        if KEEP_BEST and vl < best_val:
            best_val, best_epoch = vl, epoch + 1
            _ckpt("best.pt", slim=True)
            flag = "  <-- new best (lowest val MSE, not necessarily best-looking)"
        print(f"epoch {epoch+1:4d}: train {tl:.4f}  val {vl:.4f}  gap {vl - tl:+.4f}{flag}")

    if (epoch + 1) in MILESTONES:
        _ckpt(f"epoch_{epoch+1}.pt", slim=True)
        print(f"  milestone snapshot -> {CHECKPOINT_DIR}/epoch_{epoch+1}.pt")
    if (epoch + 1) % FULL_EVERY == 0 or epoch + 1 == NUM_EPOCHS:
        _ckpt("last.pt", slim=False)

    if PREVIEW and (epoch + 1) % PREVIEW_EVERY == 0:
        make_preview(epoch + 1)

print(f"\ntraining complete. lowest held-out MSE {best_val:.4f} at epoch {best_epoch}")
print("Compare best.pt against the milestone snapshots BY EYE — lowest val MSE is the")
print("blurriest checkpoint, not the best-looking one. Run 3 learned this the hard way.")

In [ ]:
# Plot the loss curve, with the held-out loss on the same axes.
losses = np.array(loss_history)
window = max(1, len(losses) // 100)
moving = np.convolve(losses, np.ones(window) / window, mode="valid")

plt.figure(figsize=(10, 4))
plt.plot(losses, alpha=0.2, label="per-step (train)")
plt.plot(np.arange(window - 1, len(losses)), moving, label=f"moving avg (w={window})")

if val_history:
    vs, tls, vls = zip(*val_history)
    plt.plot(vs, tls, "-", color="darkorange", lw=1, label="train (EMA, eval mode)")
    plt.plot(vs, vls, "o-", color="crimson", ms=3, label="held-out (EMA, eval mode)")
    b = int(np.argmin(vls))
    plt.axvline(vs[b], color="green", ls="--", lw=1, label=f"best val {vls[b]:.3f}")

plt.axhline(1.0, color="grey", ls=":", lw=1, label="predict-zero baseline")
plt.xlabel("step")
plt.ylabel("v-MSE loss")
plt.yscale("log")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

# What to read here. Run 2's version of this plot is the whole reason for run 3: train
# fell smoothly 0.98 -> 0.11 the entire way while held-out bottomed at ~0.62 and then
# climbed past the predict-zero baseline to 1.5157. The training curve gave no hint.
#   - train and held-out falling together  -> healthy, keep going
#   - held-out flattening while train falls -> at the limit; best.pt has the good weights
#   - held-out RISING                       -> memorizing; the run is done, stop it
if val_history:
    print(f"best held-out {min(vls):.4f} at step {vs[int(np.argmin(vls))]}, "
          f"final {vls[-1]:.4f}   (run 2 final: 1.5157)")

## Inference & evaluation checks

The training curve above says *nothing broke*. It cannot say whether the model learned
to use the text, or whether it generates anything that looks like a Pokemon. Run 1
proved that the hard way: loss fell 1.05 → 0.07 across 200 epochs and the samples were
still unusable. Everything below answers the real questions from a saved checkpoint.

Order matters — each block is only worth reading if the one before it passed:

1. **Checkpoint + EMA sanity** — is the EMA a real average, or still near its random init?
1b. **Checkpoint sweep** — which saved epoch actually generalizes best? Run 2's held-out loss bottomed at epoch ~55–100 and then rose for 500 more epochs; the last checkpoint was the worst of the run and the training curve gave no hint. Never pick by epoch number.
2. **Loss binned by timestep, train vs held-out** — *where* on the schedule did it learn, and is it memorizing?
3. **Conditioning check** — does swapping the caption change the loss at all?
4. **Samples** — the actual images.
5. **CFG sweep** — does guidance steer, or just saturate?
6. **Sampler cross-check** — do DDIM and DDPM agree?
7. **Novel prompts** — with the starting noise held fixed, so the test measures the prompt.
8. **Memorization check** — generator, or a lookup table?

Requires the cells above for `vae`, `tokenizer`, `text_encoder`, `latents_tensor`,
`embeddings_tensor`, `uncond_embedding`, `train_idx` and `val_idx`. It does **not**
require the training cell to have run in this session — it loads weights from disk and
rebuilds the architecture from the checkpoint's own `config`.

> **What run 1 measured, for comparison.** Every number below has a run-1 counterpart
> in [`RUNS.md`](../RUNS.md). The three that mattered:
>
> - Sampling from training caption #0 landed **124.68** from training latent #0 —
>   0.99× the median distance between two *random different* Pokemon. After 200
>   exposures, its output for a memorized caption was indistinguishable from an
>   unrelated one.
> - Generated per-channel latent means were **83% off**, with the spread across
>   channels collapsed to 13% of the data's. That is the zero-terminal-SNR failure,
>   and it is what per-channel normalization plus the rescaled schedule fix.
> - Best relative shuffled-minus-matched conditioning gap: **13.6%**. Alive, but thin.

In [ ]:
# 1. Load a checkpoint and check the EMA is usable.
import torch.nn.functional as F

from unet import UNet
from scheduler import NoiseScheduler
from sample import sample_ddim, sample_ddpm, latents_to_images

# TODO: point this at the checkpoint you want to evaluate.
#
# best.pt is the LOWEST-VAL-MSE checkpoint, which is not the same as the best-looking one:
# MSE is minimized by predicting the conditional mean, so it systematically selects for
# blur. Run 3's best.pt was epoch 34 of 150 and its samples were noticeably soft. Cell 1b
# scores every checkpoint; then look at the milestone snapshots too.
CKPT_PATH = "checkpoints/best.pt"

ckpt = torch.load(CKPT_PATH, map_location=device)
cfg = ckpt.get("config", {})
print(f"checkpoint : {CKPT_PATH}")
print(f"epoch      : {ckpt['epoch']}   step: {ckpt['step']}   slim: {bool(ckpt.get('slim'))}")
print(f"config     : {cfg}")

noise_scheduler = NoiseScheduler(
    zero_terminal_snr=cfg.get("zero_terminal_snr", True),
    prediction_type=cfg.get("prediction_type", "v"),
).to(device)
LATENT_SHAPE = tuple(cfg.get("latent_shape", (4, 32, 32)))
CKPT_VIEW_DIM = cfg.get("view_dim", 0)


def build_from_ckpt(state_dict):
    m = UNet(
        base_channels=cfg.get("base_channels", 64),
        num_res_blocks=cfg.get("num_res_blocks", 2),
        top_self_attn=cfg.get("top_self_attn", False),
        dropout=0.0,     # always 0 at eval
        view_dim=CKPT_VIEW_DIM,
    ).to(device)
    m.load_state_dict(state_dict)
    m.eval()
    return m


live_model = build_from_ckpt(ckpt["model"])
eval_ema = build_from_ckpt(ckpt["ema_model"])

live_flat = torch.cat([p.flatten() for p in live_model.parameters()])
ema_flat = torch.cat([p.flatten() for p in eval_ema.parameters()])
rel_delta = (ema_flat - live_flat).norm().item() / live_flat.norm().item()

print(f"\n|ema - live|/|live| = {rel_delta:.4f}   (want << 1; ~1 = EMA still near init)")
print("  run 3 read 0.0807 at epoch 34 — the EMA lag was itself a source of blur,")
print("  which is why EMA_DECAY dropped to 0.999 for this run.")

In [ ]:
# 1b. Score every saved checkpoint on train and held-out loss.
#
# Run 3's numbers: best.pt (epoch 34) val 0.5658, last.pt (epoch 150) val 0.6193. The last
# checkpoint was only 9% worse on MSE but fit far better (train 0.386 vs 0.468) — and MSE
# rewards predicting the average, so the low-MSE checkpoint is the blurry one. Use this
# cell to find DIVERGENCE, then pick between the survivors by looking at samples.
import re
from pathlib import Path

from train import validation_loss

CKPT_DIR = "checkpoints"

paths = [p for p in Path(CKPT_DIR).glob("*.pt")
         if p.stem in ("best", "last") or re.fullmatch(r"epoch_\d+", p.stem)]
paths.sort(key=lambda p: (p.stem not in ("best", "last"),
                          int(p.stem.split("_")[1]) if "_" in p.stem else 0, p.stem))
if not paths:
    raise FileNotFoundError(f"no .pt checkpoints under {CKPT_DIR}/ — check CKPT_DIR")

scored = []
print(f"{'file':>16} {'epoch':>6} {'slim':>5} {'train':>9} {'held-out':>9} {'gap':>9}")
for p in paths:
    ck = torch.load(p, map_location=device)
    c = ck.get("config", {})
    sch = NoiseScheduler(
        zero_terminal_snr=c.get("zero_terminal_snr", True),
        prediction_type=c.get("prediction_type", "v"),
    ).to(device)
    m = UNet(
        base_channels=c.get("base_channels", 64),
        num_res_blocks=c.get("num_res_blocks", 2),
        top_self_attn=c.get("top_self_attn", False),
        dropout=0.0, view_dim=c.get("view_dim", 0),
    ).to(device)
    m.load_state_dict(ck["ema_model"])   # the EMA is what we sample from
    m.eval()

    tl = validation_loss(m, sch, tr_x0, tr_ctx, view=tr_view)["weighted"]
    vl = validation_loss(m, sch, val_x0, val_ctx, view=val_view)["weighted"]
    scored.append((p, ck.get("epoch", -1), vl))
    print(f"{p.name:>16} {ck.get('epoch', -1):6d} {str(bool(ck.get('slim'))):>5} "
          f"{tl:9.4f} {vl:9.4f} {vl - tl:+9.4f}")

    del m, ck, sch
    if device == "cuda":
        torch.cuda.empty_cache()

best_path, best_ep, best_vl = min(scored, key=lambda r: r[2])
print(f"\nlowest held-out MSE: {best_path.name} (epoch {best_ep}) at {best_vl:.4f}")
if best_vl > 1.0:
    print("=> WARNING: worse than predicting zero on held-out data. Nothing generalizes;")
    print("   shrink the model or add data before reading anything into the samples.")
else:
    print("=> Sample from this AND from the latest milestone, then compare by eye.")

In [ ]:
# 2. Loss binned by timestep — train vs held-out.
#
# v-prediction makes this readable: the latents are unit-variance so E[v^2] = 1 at every t,
# and a model predicting zero scores exactly 1.0 everywhere.
#
# But 1.0 is NOT the "learned nothing" line. The optimal predictor that knows only the
# per-dim mean and variance of the training latents — no text at all — scores about 0.84.
# Judge against that. Run 3's numbers, measured:
#
#              train   held-out   vs the 0.84 no-text baseline
#   mean      0.4569     0.5717   +45.8% / +32.0%
#   t = 25    0.4983     0.5264   held-out +45%
#   t = 975   0.5838     0.8029   held-out +2.0%   <- caption->layout barely generalizes
#
# That split is the whole story: run 3 generalized well at low t (texture) and almost not
# at all at high t (global composition from the caption).

EVAL_N = 256
N_BINS = 20

cpu_gen = torch.Generator().manual_seed(1234)
tr_pool = train_idx[crop_ids[train_idx] == 0]
va_pool = val_idx[crop_ids[val_idx] == 0]
EVAL_N = min(EVAL_N, len(tr_pool), len(va_pool))


def make_eval_set(pool):
    pick = pool[torch.randperm(len(pool), generator=cpu_gen)[:EVAL_N]]
    x0 = latents_tensor[pick]
    vw = views_for_rows(pick)
    return (x0.to(device), caption_emb_for_rows(pick).to(device),
            torch.randn(x0.shape, generator=cpu_gen).to(device), pick,
            None if vw is None else vw.to(device))


tr_set = make_eval_set(tr_pool)
va_set = make_eval_set(va_pool)
print(f"eval set: {EVAL_N} train / {EVAL_N} held-out latents (crop 0)")


@torch.no_grad()
def loss_at_t(m, t_val, eval_set, ctx=None, batch=32):
    """Mean per-element MSE against the scheduler's target at one fixed timestep."""
    x0, default_ctx, noise, _, vw = eval_set
    ctx = default_ctx if ctx is None else ctx
    total, n = 0.0, 0
    for i in range(0, x0.shape[0], batch):
        xb, cb, nb = x0[i : i + batch], ctx[i : i + batch], noise[i : i + batch]
        vb = None if vw is None else vw[i : i + batch]
        t = torch.full((xb.shape[0],), t_val, device=device, dtype=torch.long)
        x_t = noise_scheduler.q_sample(xb, t, nb)
        target = noise_scheduler.get_target(xb, nb, t)
        pred = m(x_t, t, cb, vb)
        total += F.mse_loss(pred, target, reduction="sum").item()
        n += nb.numel()
    return total / n


# The no-text baseline, computed from YOUR latents rather than quoted.
_Xtr = latents_tensor[tr_pool].flatten(1).double()
_mu, _var = _Xtr.mean(0), _Xtr.var(0, unbiased=True)
_s2 = {"train": ((_Xtr - _mu) ** 2).mean(0),
       "val": ((latents_tensor[va_pool].flatten(1).double() - _mu) ** 2).mean(0)}


def no_text_baseline(t_val, which):
    a2 = noise_scheduler.alphas_cumprod[t_val].item(); b2 = 1 - a2
    return ((_s2[which] * b2 + a2 * _var ** 2) / (a2 * _var + b2) ** 2).mean().item()


bin_ts = [int((i + 0.5) * noise_scheduler.T / N_BINS) for i in range(N_BINS)]
tr_curve, va_curve, base_curve = [], [], []

print(f"\n{'t':>5} {'train v':>9} {'val v':>9} {'no-text':>9} {'val vs base':>12}")
for t_val in bin_ts:
    lt = loss_at_t(eval_ema, t_val, tr_set)
    lv = loss_at_t(eval_ema, t_val, va_set)
    bv = no_text_baseline(t_val, "val")
    tr_curve.append(lt); va_curve.append(lv); base_curve.append(bv)
    print(f"{t_val:5d} {lt:9.4f} {lv:9.4f} {bv:9.4f} {1 - lv/bv:+11.1%}")

plt.figure(figsize=(10, 4))
plt.plot(bin_ts, tr_curve, "o-", label="train (EMA)")
plt.plot(bin_ts, va_curve, "s--", label="held-out (EMA)", alpha=0.8)
plt.plot(bin_ts, base_curve, "-", color="grey", lw=1, label="no-text baseline (~0.84)")
plt.axhline(1.0, color="red", ls=":", label="predict-zero")
plt.xlabel("timestep t"); plt.ylabel("v-MSE (eval mode)")
plt.title("Denoising loss vs timestep — beat the grey line, not the red one")
plt.legend(); plt.grid(alpha=0.3); plt.show()

mt, mv, mb = float(np.mean(tr_curve)), float(np.mean(va_curve)), float(np.mean(base_curve))
print(f"\nmean   train {mt:.4f}   held-out {mv:.4f}   no-text baseline {mb:.4f}")
print(f"improvement over no-text: train {1-mt/mb:+.1%}   held-out {1-mv/mb:+.1%}"
      f"   (run 3: +45.8% / +32.0%)")
print(f"at t={bin_ts[-1]}: held-out {1-va_curve[-1]/base_curve[-1]:+.1%}   (run 3: +2.0%)"
      "   <- caption->layout generalization")

In [ ]:
# 3. Is the model actually using the text — and HOW?
#
# Same latents, same noise, same t; only the caption changes.
#   matched  = each latent with its own caption
#   shuffled = each latent with someone else's caption
#   uncond   = the "" embedding
#   stripped = its own caption with the Pokemon NAME replaced by "creature"
#
# The last two are the interesting pair. Run 2 read as spectacularly conditioned — a
# 6702% relative matched-vs-shuffled gap — but that was a LOOKUP TABLE, not comprehension.
# The tell: uncond (0.3380) beat shuffled (2.1158) by 6x. Given no caption it produced
# something generic and sane; given a WRONG one it confidently produced the wrong image.
#
# `stripped` is the new probe. If stripped is close to matched, the model is reading the
# DESCRIPTION. If stripped collapses toward uncond, it was keying on the name.

shuf = torch.randperm(EVAL_N, generator=torch.Generator().manual_seed(7))
assert (shuf == torch.arange(EVAL_N)).sum().item() < EVAL_N * 0.05

tr_rows = tr_set[3]
tr_ctx_m = tr_set[1]
shuffled_ctx = tr_ctx_m[shuf.to(tr_ctx_m.device)]
uncond_ctx = uncond_embedding.to(device).expand(EVAL_N, -1, -1)
has_variants = stats["caption_variants"] > 1
stripped_ctx = caption_emb_for_rows(tr_rows, variant=1).to(device) if has_variants else None

hdr = f"{'t':>5}  {'matched':>9}  {'shuffled':>9}  {'uncond':>9}"
if has_variants:
    hdr += f"  {'stripped':>9}"
hdr += f"  {'shuf rel':>9}"
print(hdr)

rel_gaps = []
for t_val in [50, 150, 300, 500, 700, 900]:
    l_m = loss_at_t(eval_ema, t_val, tr_set)
    l_s = loss_at_t(eval_ema, t_val, tr_set, ctx=shuffled_ctx)
    l_u = loss_at_t(eval_ema, t_val, tr_set, ctx=uncond_ctx)
    rel_gaps.append((l_s - l_m) / max(l_m, 1e-12))
    line = f"{t_val:5d}  {l_m:9.4f}  {l_s:9.4f}  {l_u:9.4f}"
    if has_variants:
        l_v = loss_at_t(eval_ema, t_val, tr_set, ctx=stripped_ctx)
        line += f"  {l_v:9.4f}"
    line += f"  {(l_s - l_m) / max(l_m, 1e-12):8.1%}"
    print(line)

best = max(rel_gaps)
print(f"\nlargest RELATIVE shuffled-minus-matched gap: {best:.1%}   (run 1: 13.6%   run 2: 6702%)")
if best < 0.01:
    print("=> CONDITIONING IS DEAD. The caption does not change the prediction.")
elif best < 0.05:
    print("=> WEAK conditioning. Expect prompt-insensitive samples.")
elif best > 5.0:
    print("=> Suspiciously strong. Check `uncond` vs `shuffled` above: if uncond is much")
    print("   LOWER, this is caption->image retrieval, not text understanding.")
else:
    print("=> Conditioning is live and in a healthy range.")

In [ ]:
# 4. Helpers: encode arbitrary prompts, generate, and show a row of decoded samples.

CFG = 2.0            # TODO: set from the sweep two cells down
CFG_RESCALE = 0.7    # Lin et al. section 3.4 — counteracts CFG over-exposure
CLIP_X0 = 4.3        # TODO: re-derive from the print at the end of this cell


@torch.no_grad()
def encode_prompt(prompts):
    """list[str] -> (B, 77, 768) CLIP embeddings, same pipeline as the training data."""
    if isinstance(prompts, str):
        prompts = [prompts]
    ids = tokenizer(
        prompts, padding="max_length", max_length=77, truncation=True, return_tensors="pt",
    ).input_ids.to(device)
    return text_encoder(ids).last_hidden_state


@torch.no_grad()
def generate(cond_emb, method="ddim", num_steps=50, guidance_scale=None,
             guidance_rescale=None, eta=0.0, seed=42, fixed_noise=False, view=None):
    """
    Sample latents and return (images in [0,1], raw latents). Uses the EMA model.

    view defaults to the CANONICAL view (full frame, unflipped) — the whole point of the
    micro-conditioning is that you can ask for one specific framing instead of getting the
    average of all 8.

    fixed_noise=True gives every row the SAME starting noise; use it whenever you compare
    PROMPTS, or the difference you measure is mostly the noise draw.
    """
    gs = CFG if guidance_scale is None else guidance_scale
    gr = CFG_RESCALE if guidance_rescale is None else guidance_rescale
    g = torch.Generator(device=device).manual_seed(seed)

    B = cond_emb.shape[0]
    x_T = None
    if fixed_noise:
        x_T = torch.randn(1, *LATENT_SHAPE, device=device, generator=g).expand(B, -1, -1, -1).contiguous()
    if view is None and CKPT_VIEW_DIM > 0:
        view = canonical_views(B).to(device)

    kw = dict(guidance_scale=gs, guidance_rescale=gr, num_steps=num_steps,
              latent_shape=LATENT_SHAPE, clip_x0=CLIP_X0, x_T=x_T, view=view, generator=g)
    if method == "ddim":
        lat = sample_ddim(eval_ema, noise_scheduler, cond_emb,
                          uncond_embedding.to(device), eta=eta, **kw)
    else:
        lat = sample_ddpm(eval_ema, noise_scheduler, cond_emb,
                          uncond_embedding.to(device), **kw)

    imgs = latents_to_images(lat, vae, lat_mean=LAT_MEAN, lat_std=LAT_STD).cpu()
    return imgs, lat


def show_row(images, titles, suptitle=None, wrap=28):
    """images: (B, 3, H, W) in [0, 1]."""
    n = images.shape[0]
    fig, axes = plt.subplots(1, n, figsize=(3.2 * n, 3.8))
    axes = [axes] if n == 1 else list(axes)
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img.permute(1, 2, 0).clamp(0, 1).numpy())
        wrapped = "\n".join(str(title)[i : i + wrap] for i in range(0, min(len(str(title)), wrap * 3), wrap))
        ax.set_title(wrapped, fontsize=8)
        ax.axis("off")
    if suptitle:
        fig.suptitle(suptitle, fontsize=11)
    plt.tight_layout()
    plt.show()


# THREE references. A single lumped std hid a real defect in run 2: its samples reported
# batch std 1.0096 against a batch reference of 1.000, which looked perfect. Split apart,
# per-sample std was 0.706 vs a 0.987 reference (28% UNDER-dispersed, i.e. over-smoothed)
# while brightness drifted 5.9x too much. The two errors cancelled in the lumped number.
_REF_BATCH_STD = latents_tensor.std().item()
_REF_SAMPLE_STD = latents_tensor.flatten(1).std(dim=1).mean().item()
_REF_MEAN_SPREAD = latents_tensor.flatten(1).mean(dim=1).std().item()


def latent_stats(name, lat):
    """Compare generated latent statistics against the training latents, decomposed."""
    flat = lat.flatten(1)
    per_sample = flat.std(dim=1).mean().item()
    pc_mean = lat.mean(dim=(0, 2, 3)).tolist()
    line = (f"{name:20s} per-sample std {per_sample:.3f} "
            f"(ref {_REF_SAMPLE_STD:.3f}, {per_sample / _REF_SAMPLE_STD - 1:+.0%})")
    if lat.shape[0] > 1:
        spread = flat.mean(dim=1).std().item()
        line += (f"  mean-spread {spread:.3f} (ref {_REF_MEAN_SPREAD:.3f}, "
                 f"{spread / max(_REF_MEAN_SPREAD, 1e-9):.1f}x)")
    line += f"  per-chan mean {[round(v, 2) for v in pc_mean]}"
    print(line)


print(f"TRAINING reference   per-chan mean={[round(v, 3) for v in latents_tensor.mean(dim=(0,2,3)).tolist()]}")
print(f"                     batch std {_REF_BATCH_STD:.3f}   per-sample std {_REF_SAMPLE_STD:.3f}   "
      f"mean-spread {_REF_MEAN_SPREAD:.3f}")

# torch.quantile caps out at ~2**24 elements and 6,664 x 4,096 = 27.3M exceeds it, so use
# kthvalue — exact, no size limit. This is what crashed run 3's eval section.
_a = latents_tensor.abs().flatten()
q = _a.kthvalue(int(0.9999 * _a.numel())).values.item()
print(f"\nsuggested CLIP_X0 = {q:.1f}   (99.99th pct of |x_0|; abs max {_a.max().item():.1f})")
del _a

In [ ]:
# 5. THE inference check: sample from captions the model actually trained on.
#
# The easiest possible ask. If these do not look like Pokemon, nothing else will.
#
# Run 2 aced this and it was bad news — 3 of 4 were near-pixel copies at latent distance
# 11.66-16.90 against a p1 threshold of 68.13. What you want here is recognisable but NOT
# identical, with the memorization cell at the end confirming the distances.
#
# Image 0's caption is unusually templated ("A cheerful X ready for its next adventure")
# and was run 2's one retrieval failure, so start at 1.
SHOW_IMAGES = [1, 2, 3, 4]
captions = [poke_data["train"][i]["text"] for i in SHOW_IMAGES]
cond = caption_emb(SHOW_IMAGES).to(device)

imgs, lats = generate(cond, method="ddim", num_steps=50, seed=SEED)
latent_stats(f"generated (cfg {CFG})", lats)
show_row(imgs, captions, suptitle=f"DDIM 50, CFG {CFG} (rescale {CFG_RESCALE}) — training captions")

real = torch.stack([preprocess(poke_data["train"][i]["image"]) for i in SHOW_IMAGES])
show_row((real + 1) / 2, captions, suptitle="Ground truth for the same captions")

In [ ]:
# 6. CFG sweep — same seed, same prompt, varying guidance.
#
# Run 2's measured latent statistics, correlating the 4 generated channel means against
# the data's, WITHOUT rescale:
#
#   CFG    std     per-chan means                  verdict
#   1.0    0.72    [0.16, 0.17, 0.36, 0.16]        fine
#   5.0    1.08    [0.63, 0.58, -0.02, -0.03]      drifting
#   7.5    1.62    [1.03, 0.63, -0.73, -0.21]      blown out, pattern inverted
#
# With rescale=0.7 the same sweep held std at 0.71-0.88 all the way to 7.5. So the rescale
# works; the useful question is which w gives prompt adherence without mush.
#
# Also a conditioning check by another route: identical columns mean pred_cond ==
# pred_uncond and the guidance term is doing nothing.
CFG_IMAGE = 1          # NOT 0 — image 0's caption was run 2's retrieval failure, so
                       # sweeping on it made the model look far worse than it was.
cfg_caption = poke_data["train"][CFG_IMAGE]["text"]
cfg_cond = caption_emb([CFG_IMAGE]).to(device)

print(f"prompt: {cfg_caption}\n")
scales = [1.0, 1.5, 2.0, 3.0, 5.0, 7.5]

for rescale in [0.0, CFG_RESCALE]:
    cfg_imgs = []
    print(f"--- guidance_rescale = {rescale} ---")
    for w in scales:
        img, lat = generate(cfg_cond, method="ddim", num_steps=50, guidance_scale=w,
                            guidance_rescale=rescale, seed=SEED)
        latent_stats(f"cfg {w}", lat)
        cfg_imgs.append(img[0])
    show_row(torch.stack(cfg_imgs), [f"CFG {w}" for w in scales],
             suptitle=f"Guidance sweep, rescale={rescale} (same seed)")

In [ ]:
# 7. Sampler cross-check: DDIM step count, DDIM vs DDPM, and eta.
#
# Both samplers reverse the SAME trained model, so they should agree. Disagreement
# localises the problem:
#   - DDPM(1000) fine but DDIM(50) broken -> the strided alpha_bar grid math is wrong
#   - both broken but per-timestep loss healthy -> a sampler bug, not training
#   - DDIM(50) slightly worse than DDIM(250) -> discretization error, expected
#
# Run 2 came back clean here: all five variants within ~3.6% on every latent statistic
# (std 0.6996 to 0.7251). Run 1's DDIM-drifts-to-the-mean behaviour was gone. So this cell
# is now a regression check rather than a live suspicion — if it stays tight, sampling is
# not your problem.
sweep_cond = caption_emb([CFG_IMAGE]).to(device)

variants = [
    ("ddim", 25, 0.0), ("ddim", 50, 0.0), ("ddim", 250, 0.0),
    ("ddim", 50, 1.0), ("ddpm", 1000, 0.0),
]

sweep_imgs, sweep_titles = [], []
for method, steps, eta in variants:
    img, lat = generate(sweep_cond, method=method, num_steps=steps, eta=eta, seed=SEED)
    label = f"{method} {steps}" + (f" eta{eta}" if eta else "")
    latent_stats(label, lat)
    sweep_imgs.append(img[0])
    sweep_titles.append(label.upper())

show_row(torch.stack(sweep_imgs), sweep_titles,
         suptitle=f"Sampler comparison (CFG {CFG}, same seed)")

In [ ]:
# 8. Novel prompts — the only test of generalization the samples can give.
#
# Captions the model has never seen. fixed_noise=True pins the starting noise across
# rows, so every difference below is attributable to the prompt.
#
# Run 2 got COLOUR right on all four (red / blue / yellow / green) and structure wrong on
# all four — amorphous blobs. That is exactly the ceiling of a lookup table asked to
# extrapolate, and it is why run 3 adds name-stripped caption variants: the descriptive
# channel already carries signal, it just was not the cheapest route to a low training
# loss. Progress here is the single best sign run 3 worked.
novel = [
    "a red fire dragon pokemon with large wings",
    "a blue water turtle pokemon with a hard shell",
    "a small yellow electric mouse pokemon",
    "a green grass pokemon with a flower on its back",
]
novel_cond = encode_prompt(novel)

novel_imgs, novel_lats = generate(novel_cond, method="ddim", num_steps=50,
                                  seed=SEED, fixed_noise=True)
latent_stats("generated (novel)", novel_lats)
show_row(novel_imgs, novel, suptitle=f"Unseen prompts, SHARED starting noise (DDIM 50, CFG {CFG})")

flat = novel_lats.flatten(1)
dist = torch.cdist(flat, flat).cpu()
print("\npairwise latent L2 between samples (same x_T, so this is prompt effect only):")
print(dist.numpy().round(2))
off = dist[~torch.eye(len(novel), dtype=bool)]
print(f"mean off-diagonal: {off.mean().item():.2f}")

# Reference: same prompts, INDEPENDENT noise. Run 2 scored 100.2% here, meaning x_T had
# no effect at all — one prompt, one image, zero diversity. A healthy model should sit
# well below 100%: the prompt should matter, but so should the noise.
_, indep_lats = generate(novel_cond, method="ddim", num_steps=50, seed=SEED, fixed_noise=False)
iflat = indep_lats.flatten(1)
idist = torch.cdist(iflat, iflat).cpu()
ioff = idist[~torch.eye(len(novel), dtype=bool)]
print(f"same prompts, independent noise: mean off-diagonal {ioff.mean().item():.2f}")
print(f"prompt effect / total variation : {off.mean().item() / ioff.mean().item():.1%}   (run 2: 100.2%)")

In [ ]:
# 9. Memorization check — generator, or a lookup table?
#
# For each generated latent, find the nearest TRAINING latent and look at them together.
# Thresholds are recomputed at runtime; run 2's absolute numbers do not transfer because
# the augmented set has different geometry.
#
# Use p1, NOT min: with mirrored and cropped views in the pool, `min` is driven by
# near-symmetric Pokemon whose mirror is almost identical to themselves. Run 2's min fell
# from 18.77 (833 originals) to 8.99 for exactly that reason, which makes it useless as a
# copy threshold.
#
# Run 2 scored 0.14-0.20x median with a train/held-out nearest ratio of 0.345 — a copier.
# Healthy is ~1.0x median and a ratio near 1.0.

# Compare against crop-0 training rows only, so distances are not flattered by having a
# random crop that happens to sit closer.
mem_rows = train_idx[crop_ids[train_idx] == 0]
train_flat = latents_tensor[mem_rows].flatten(1)
train_d = torch.cdist(train_flat, train_flat)
off_diag = train_d[~torch.eye(train_flat.shape[0], dtype=torch.bool)]
P1 = off_diag.quantile(0.01).item()
MINDIST = off_diag.min().item()
MEDIAN = off_diag.median().item()

print(f"train-vs-train latent L2 ({len(mem_rows)} crop-0 rows): mean {off_diag.mean():.2f}  "
      f"median {MEDIAN:.2f}  p1 {P1:.2f}  min {MINDIST:.2f}")
print(f"  -> suspicious below {P1:.2f}, unambiguous copy below {MINDIST:.2f}\n")

gen_imgs, gen_lats = generate(caption_emb(SHOW_IMAGES).to(device), num_steps=50, seed=SEED)
d = torch.cdist(gen_lats.flatten(1).cpu(), train_flat)
nn_dist, nn_pos = d.min(dim=1)
nn_rows = mem_rows[nn_pos]

for i, (row, dist_i) in enumerate(zip(nn_rows.tolist(), nn_dist.tolist())):
    img_id = int(group_ids[row])
    tag = " (mirrored)" if int(stats["flip_ids"][row]) else ""
    flag = "  <-- COPY" if dist_i < MINDIST else ("  <-- suspicious, inspect" if dist_i < P1 else "")
    print(f"sample {i}: nearest = image {img_id:3d}{tag}  dist {dist_i:7.2f}  "
          f"({dist_i / MEDIAN:.2f}x median){flag}")

# Closer to train than to held-out? A big asymmetry is memorization even when no single
# distance trips the threshold above.
val_flat = latents_tensor[val_idx[crop_ids[val_idx] == 0]].flatten(1)
d_val = torch.cdist(gen_lats.flatten(1).cpu(), val_flat).min(dim=1).values
print(f"\nmean nearest to TRAIN {nn_dist.mean():.2f}   to HELD-OUT {d_val.mean():.2f}"
      f"   ratio {nn_dist.mean() / d_val.mean():.3f}  (want ~1.0, run 2: 0.345)")

nn_real = []
for row in nn_rows.tolist():
    im = preprocess(poke_data["train"][int(group_ids[row])]["image"])
    nn_real.append(torch.flip(im, dims=[-1]) if int(stats["flip_ids"][row]) else im)
show_row(gen_imgs, [f"generated {i}" for i in range(len(nn_rows))], suptitle="Generated")
show_row((torch.stack(nn_real) + 1) / 2,
         [f"nearest image #{int(group_ids[r])}" for r in nn_rows.tolist()],
         suptitle="Nearest training latent (crop 0)")

In [ ]:
# finished